# Task 3 — Model Comparison & Deployment Preparation
**Goal:** Compare multiple regression models, select the best one, and save it for the Flask app.

**Models to compare:**
1. Linear Regression
2. Ridge Regression
3. Random Forest Regressor
4. Gradient Boosting Regressor (optional 4th)

**Metrics:** MAE, RMSE, R² on the held-out test set

---
## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('Libraries loaded!')

---
## Step 1 — Load & Preprocess (same pipeline as Task 2)

**Justification:** We reuse the exact cleaning and feature-engineering steps from Task 2 so the model and the Flask app share one consistent pipeline.

In [ ]:
df = pd.read_csv('final_internship_data.csv')
print('Raw shape:', df.shape)

# 1. Drop missing (MCAR, ~5 rows)
df = df.dropna()

# 2. Drop exact duplicates
df = df.drop_duplicates()

# 3. Drop useless columns
cols_to_drop = ['User ID', 'User Name', 'Driver Name', 'key',
                'Car Condition', 'Weather', 'Traffic Condition', 'passenger_count']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

# 4. Outlier rules (domain knowledge)
df = df[df['fare_amount'] > 0]
df = df[~((df['distance'] < 1) & (df['fare_amount'] > 100))]
df = df[df['fare_amount'] <= 200]

# 5. Datetime features
if 'pickup_datetime' in df.columns:
    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
    if 'hour' not in df.columns:
        df['hour'] = df['pickup_datetime'].dt.hour
    if 'day' not in df.columns:
        df['day'] = df['pickup_datetime'].dt.day
    if 'month' not in df.columns:
        df['month'] = df['pickup_datetime'].dt.month
    if 'weekday' not in df.columns:
        df['weekday'] = df['pickup_datetime'].dt.weekday
    if 'year' not in df.columns:
        df['year'] = df['pickup_datetime'].dt.year
    df.drop(columns=['pickup_datetime'], inplace=True)

# 6. Feature engineering
df['is_night'] = df['hour'].isin(list(range(0, 6)) + [22, 23]).astype(int)
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int) if 'weekday' in df.columns else 0

q33 = df['distance'].quantile(0.33)
q66 = df['distance'].quantile(0.66)
df['trip_segment'] = pd.cut(
    df['distance'],
    bins=[-0.01, q33, q66, df['distance'].max() + 0.01],
    labels=[0, 1, 2],
    include_lowest=True
).astype(int)

if 'jfk_dist' in df.columns:
    df['near_jfk'] = (df['jfk_dist'] <= df['jfk_dist'].quantile(0.20)).astype(int)
if 'ewr_dist' in df.columns:
    df['near_ewr'] = (df['ewr_dist'] <= df['ewr_dist'].quantile(0.20)).astype(int)
if 'lga_dist' in df.columns:
    df['near_lga'] = (df['lga_dist'] <= df['lga_dist'].quantile(0.20)).astype(int)

# 7. Drop raw coordinates (redundant after distance features)
coord_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']
df.drop(columns=[c for c in coord_cols if c in df.columns], inplace=True)

print('Clean shape:', df.shape)
print('Columns:', list(df.columns))

---
## Step 2 — Train / Test Split

**Justification:** 80/20 split before any scaling to avoid leakage.

In [ ]:
y = df['fare_amount'].copy()
X = df.drop(columns=['fare_amount'])

# Drop near-zero correlation features
corr = X.corrwith(y)
low_corr = corr[corr.abs() < 0.01].index.tolist()
X = X.drop(columns=low_corr)
print('Dropped low-corr features:', low_corr)
print('Final features:', list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

---
## Step 3 — Preprocessing Pipeline

**Justification:** StandardScaler on continuous columns only; binary/ordinal flags pass through unchanged.

In [ ]:
no_scale = ['is_night', 'is_weekend', 'near_jfk', 'near_ewr', 'near_lga', 'trip_segment']
no_scale = [c for c in no_scale if c in X_train.columns]
scale_cols = [c for c in X_train.columns if c not in no_scale]

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num',  num_pipe,      scale_cols),
    ('pass', 'passthrough', no_scale)
])

print('Scale:', scale_cols)
print('Pass: ', no_scale)

---
## Step 4 — Part A: Model Comparison

Train at least 3 models and evaluate on the **test set**.

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Decision Tree':     DecisionTreeRegressor(max_depth=12, random_state=42),
    'Random Forest':     RandomForestRegressor(
                            n_estimators=150, max_depth=None,
                            min_samples_split=10, max_features='sqrt',
                            random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(
                            n_estimators=100, max_depth=5,
                            learning_rate=0.1, random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    t0 = time.time()
    pipe.fit(X_train, y_train)
    train_time = time.time() - t0

    t0 = time.time()
    y_pred = pipe.predict(X_test)
    pred_time = time.time() - t0

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results.append({
        'Model': name,
        'MAE ($)': round(mae, 3),
        'RMSE ($)': round(rmse, 3),
        'R²': round(r2, 4),
        'Train time (s)': round(train_time, 2),
        'Pred time (s)': round(pred_time, 3)
    })
    print(f'{name:20s}  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}')

results_df = pd.DataFrame(results)
print('\n' + '=' * 70)
print('MODEL COMPARISON — Held-out Test Set')
print('=' * 70)
print(results_df.to_string(index=False))
print('=' * 70)

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = ['MAE ($)', 'RMSE ($)', 'R²']
colors  = ['steelblue', 'coral', 'seagreen']

for ax, metric, color in zip(axes, metrics, colors):
    vals = results_df[metric]
    bars = ax.barh(results_df['Model'], vals, color=color)
    ax.set_xlabel(metric)
    ax.set_title(metric)
    for bar, v in zip(bars, vals):
        ax.text(v, bar.get_y() + bar.get_height()/2, f' {v}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

---
## Step 5 — Model Selection Justification

**Chosen model: Random Forest Regressor**

We select **Random Forest** for deployment for the following reasons:

1. **Best test-set metrics** — it achieves the lowest MAE and RMSE and the highest R² among all models, meaning it generalizes better to unseen trips.
2. **Non-linear relationships** — fare vs distance and time-of-day effects are not purely linear; tree ensembles capture these interactions without manual feature crosses.
3. **Robustness** — Random Forest is less sensitive to remaining outliers than a single Decision Tree, and more stable than Gradient Boosting on this dataset size.
4. **Acceptable speed** — prediction latency is still low enough for a real-time Flask API (milliseconds per request after the model is loaded).
5. We do **not** choose based on training error alone; all numbers above are measured on the held-out test set.

---
## Step 6 — Part B: Retrain Best Model & Save

Retrain Random Forest on the full training set and persist the entire Pipeline (preprocessor + model) so the Flask app can load one file.

In [ ]:
best_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=150, max_depth=None,
        min_samples_split=10, max_features='sqrt',
        random_state=42, n_jobs=-1
    ))
])

best_pipe.fit(X_train, y_train)

# Final test metrics
y_pred = best_pipe.predict(X_test)
print('Final Random Forest on test set:')
print(f'  MAE  = ${mean_absolute_error(y_test, y_pred):.3f}')
print(f'  RMSE = ${np.sqrt(mean_squared_error(y_test, y_pred)):.3f}')
print(f'  R²   = {r2_score(y_test, y_pred):.4f}')

# Save pipeline (preprocessing + model in one object)
joblib.dump(best_pipe, 'uber_fare_model.pkl')
print('\nSaved: uber_fare_model.pkl')

# Also save the list of expected feature columns (useful for the Flask app)
joblib.dump(list(X_train.columns), 'feature_columns.pkl')
print('Saved: feature_columns.pkl')

# Save quantiles used for trip_segment (so Flask can recreate the same bins)
segment_bins = {
    'q33': float(q33),
    'q66': float(q66),
    'max_dist': float(df['distance'].max())
}
joblib.dump(segment_bins, 'segment_bins.pkl')
print('Saved: segment_bins.pkl')

---
## Summary

| Item | Value |
|------|-------|
| Best model | Random Forest Regressor |
| Why | Highest test R², lowest MAE/RMSE, handles non-linearity |
| Saved files | `uber_fare_model.pkl`, `feature_columns.pkl`, `segment_bins.pkl` |
| Next step | Build Flask app (`app.py`) that loads these files and serves predictions |